In [15]:
from langgraph.graph import StateGraph , END, START
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [6]:
load_dotenv()

True

In [4]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [7]:
model = ChatGroq(model= "llama-3.1-8b-instant")

In [8]:
def chat_node(state: ChatState):
    messages= state['messages']

    response=model.invoke(messages).content

    return {'messages': [response]}

In [18]:
checkpointer= InMemorySaver()
graph= StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node',END)

chatbot=graph.compile(checkpointer=checkpointer)

In [20]:
thread_id='1'
while True:
    user_message=input('type here: ')
    print("User ",user_message)
    if user_message.strip().lower() in ['exit','quit','bye']:
        break
    config= {'configurable':{'thread_id':thread_id}}
    response=chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print("AI ", response['messages'][-1].content)
    

User  hi
AI  Hello. How can I help you today?
User  my name is gaurav
AI  Hello Gaurav. It's nice to meet you. I'm here to help answer any questions or discuss topics you're interested in. What's on your mind today?
User  now tell me my name
AI  Your name is Gaurav.
User  okay do you know about the meti japan internship 
AI  The METI Japan Internship is a great program. The Ministry of Economy, Trade, and Industry (METI) of Japan offers internships to international students, usually graduate students, as part of its initiative to promote international exchange and cooperation.

The internship provides an opportunity for students to gain hands-on experience in various fields related to economics, trade, and industry, such as research, policy-making, and business operations. The program is highly competitive, and applicants typically need to meet specific requirements, such as being a graduate student, having a strong academic record, and proficiency in Japanese.

If you're interested in